# 0.1 - Python for GenAI: The 8 Skills You Actually Need

**Module 0 - Python Prerequisites** | Netsetos GenAI Engineering

This notebook is a hands-on tour of the six Python patterns every GenAI app leans on: async concurrency, generators/streaming, Pydantic validation, decorators, env vars + JSON parsing, and resilient error handling. Every cell runs against a stubbed LLM (fake sleeps and hardcoded responses, no real API key), so you can watch the mechanics in isolation before wiring them to a real model. The final cell fuses all of them into the exact shape of a production LLM call.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

One line that sets the tone for the whole notebook: the examples use seeded/deterministic values so runs are reproducible. There are no package installs or imports here - each demo cell imports exactly what it needs at the top.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A note-to-self comment cell, not executable logic. It flags that the lesson leans on fixed/seeded values so your output matches the walkthroughs, and that imports live inside each individual demo rather than being front-loaded here.

**How the code works - step by step**
- **The comment** - documents the reproducibility intent for the reader; nothing runs.
- **No imports** - deliberate; `asyncio`, `time`, `pydantic`, etc. are imported inside the cell that uses them so each demo stands alone.

**In one line:** a placeholder/reproducibility marker - the real work starts in the next cell.

**What you'll see:** (no output - environment prep)

## 1 - Async/await: run 5 LLM calls in parallel

LLM calls are slow (2-5s each) and I/O-bound - your code spends that time waiting on the network, not computing. `async`/`await` lets one thread fire all the calls, then wait for them together instead of one-after-another. This cell times the sequential version against the parallel version so you can see the speedup, not just be told about it.

In [ ]:
# =============================================
# parallel message summarization
# Sequential: 10 seconds. Async: 2 seconds.
# =============================================

import asyncio, time

# Simulate an LLM API call (takes 2 seconds)
async def summarize_message(message: str, msg_id: int) -> str:
    print(f"  Msg {msg_id}: summarizing '{message[:30]}...'")
    await asyncio.sleep(2)  # simulated network wait
    return f"Summary of msg {msg_id}"

# ## Sequential (slow) ##
async def sequential():
    start = time.time()
    messages = ["What's a transformer?", "Explain RAG", "What is fine-tuning?", "Define embeddings", "What's an agent?"]
    results = []
    for i, m in enumerate(messages):
        r = await summarize_message(m, i+1)
        results.append(r)
    print(f"  Sequential: {time.time()-start:.1f}s for {len(results)} calls\n")

# ## Async (fast) ##
async def parallel():
    start = time.time()
    messages = ["What's a transformer?", "Explain RAG", "What is fine-tuning?", "Define embeddings", "What's an agent?"]
    tasks = [summarize_message(m, i+1) for i, m in enumerate(messages)]
    results = await asyncio.gather(*tasks)  # magic line
    print(f"  Async:      {time.time()-start:.1f}s for {len(results)} calls")

print("Sequential:")
asyncio.run(sequential())
print("Parallel:")
asyncio.run(parallel())

# Output:
# Sequential:
#   Msg 1: summarizing 'What's a transformer?...'
#   Msg 2: summarizing 'Explain RAG...'
#   ... (5 lines printed one every 2s)
#   Sequential: 10.2s for 5 calls
# Parallel:
#   Msg 1..5: (all printed in <0.01s)
#   Async:      2.1s for 5 calls   # ~4.9x speedup

**Explanation**

A side-by-side timing benchmark of the same 5 fake LLM calls run two ways. `summarize_message` stubs a real API call with `await asyncio.sleep(2)`. The point isn't the summaries - it's the wall-clock difference between awaiting calls in a loop versus gathering them concurrently.

**How the code works - step by step**
- **`summarize_message`** - an `async` coroutine that prints, `await asyncio.sleep(2)` to fake the network wait, and returns a summary string.
- **`sequential()`** - loops over 5 messages and `await`s each one before starting the next, so the sleeps stack: 5 x 2s.
- **`parallel()`** - builds a list of coroutines first, then `await asyncio.gather(*tasks)` launches them all at once and waits for the slowest (2s).
- **The two `asyncio.run(...)` calls** - drive each version and print its elapsed time via `time.time()`.

**In one line:** same 5 calls, one loop stacks the waits, `gather` overlaps them.

**What you'll see:** Sequential prints its 5 messages one every 2 seconds and reports ~10.2s; the parallel run prints all 5 messages nearly instantly and reports ~2.1s - roughly a 5x speedup.

## 2 - Generators & yield: stream tokens as they arrive

LLMs produce output one token at a time. A generator (`yield`) hands each piece to the caller the instant it's ready instead of building the whole answer and returning it at the end. This cell contrasts a normal function that makes you wait for everything with a generator that lets words appear progressively - the mechanism behind ChatGPT-style streaming.

In [ ]:
# =============================================
# stream tokens to the user as they arrive
# yield sends one item at a time instead of
# waiting to build the entire response.
# =============================================

import time

# ## Normal function: waits until EVERYTHING is ready ##
def demo_reply_normal() -> str:
    words = ["The", "future", "of", "AI", "is", "bright"]
    result = []
    for word in words:
        time.sleep(0.3)  # simulate generation time
        result.append(word)
    return " ".join(result)  # user waits 1.8s, then sees everything

# ## Generator: streams word by word ##
def reply_stream():
    words = ["The", "future", "of", "AI", "is", "bright"]
    for word in words:
        time.sleep(0.3)
        yield word  # sends each word IMMEDIATELY

# Normal: user sees nothing for 1.8s, then everything
print("Normal function:")
start = time.time()
result = demo_reply_normal()
print(f"  {result} (waited {time.time()-start:.1f}s)\n")

# Generator: user sees words appear one by one
print("Generator (streaming):")
start = time.time()
print("  ", end="")
for word in reply_stream():
    print(word, end=" ", flush=True)  # appears immediately
print(f"\n  (first word at 0.3s, total {time.time()-start:.1f}s)")

# Output:
# Normal function:
#   The future of AI is bright (waited 1.8s)
# Generator (streaming):
#   The future of AI is bright
#   (first word at 0.3s, total 1.9s)

**Explanation**

A perceived-latency demo. Both versions produce the identical six-word sentence in the same total time; the difference is *when* the user sees the first word. It shows that `return` forces a wait-for-everything while `yield` emits incrementally.

**How the code works - step by step**
- **`demo_reply_normal`** - loops over the words, `time.sleep(0.3)` each, appends to a list, and `return`s the joined string only after all six are done.
- **`reply_stream`** - same loop and sleeps, but `yield`s each word immediately, making it a generator instead of a plain function.
- **First print block** - calls the normal function and shows the full result after the total wait.
- **Second print block** - iterates the generator, printing each word with `flush=True` so it appears the moment it's yielded.

**In one line:** `return` = wait then dump everything; `yield` = emit as you go.

**What you'll see:** The normal function shows nothing for ~1.8s then prints the whole sentence at once; the generator prints words one by one starting at ~0.3s, finishing around 1.9s total. Same content, first word visible ~6x sooner.

## 3 - Pydantic: validate messy LLM JSON

Models return text that's *usually* valid JSON but often violates your contract - a confidence of `95` instead of `0.92`, an uppercase `"QUESTION"`, a bool sent as `"maybe"`. Pydantic v2 lets you declare the exact shape once and rejects anything that doesn't fit, on the first call. This cell defines a schema and runs both good and deliberately broken model output through it.

> **Needs Pydantic v2** (`pip install pydantic>=2.13`)

In [ ]:
# =============================================
# message classifier - validate LLM JSON
# Pydantic v2 (pip install pydantic>=2.13)
# =============================================

from pydantic import BaseModel, Field
from typing import Literal, Optional
import json

# Define WHAT the LLM output should look like
class MessageClassification(BaseModel):
    category: Literal["question", "command", "smalltalk", "complaint", "other"]
    confidence: float = Field(ge=0, le=1, description="Confidence 0-1")
    rationale: str = Field(max_length=200)
    needs_followup: bool

# Simulate LLM returning JSON text
llm_output = '{"category": "question", "confidence": 0.92, "rationale": "user asks about a topic", "needs_followup": true}'

# Parse and validate - Pydantic catches errors automatically
result = MessageClassification.model_validate_json(llm_output)
print(f"Category:    {result.category}")
print(f"Confidence:  {result.confidence:.0%}")
print(f"Needs reply: {result.needs_followup}")
print(f"Type check:  confidence is {type(result.confidence).__name__}")

# What if the LLM returns garbage?
print("\nBad LLM output:")
try:
    bad = MessageClassification.model_validate_json(
        '{"category": "QUESTION", "confidence": 95, "rationale": "x", "needs_followup": "maybe"}'
    )
except Exception as e:
    print(f"  Pydantic caught it:")
    print(f"  - 'QUESTION' not in Literal categories (must be lowercase)")
    print(f"  - confidence 95 is > le=1")
    print(f"  - 'maybe' is not a valid bool")

# Output:
# Category:    question
# Confidence:  92%
# Needs reply: True
# Type check:  confidence is float
# 
# Bad LLM output:
#   Pydantic caught it:
#   - 'QUESTION' not in Literal categories (must be lowercase)
#   - confidence 95 is > le=1
#   - 'maybe' is not a valid bool

**Explanation**

A schema-as-contract demo. `MessageClassification` is a `BaseModel` whose field types and constraints (`Literal`, `Field(ge=0, le=1)`, `max_length`, `bool`) encode exactly what valid output looks like. `model_validate_json` parses and validates in one step - the interesting part is watching it accept clean JSON and reject garbage.

**How the code works - step by step**
- **`class MessageClassification(BaseModel)`** - declares four fields: a `Literal` of five lowercase categories, a `confidence` float constrained to 0-1, a `rationale` capped at 200 chars, and a `needs_followup` bool.
- **First `model_validate_json`** - parses well-formed JSON; the fields come back as typed Python (`confidence` is a real `float`, printed as a percentage).
- **The `try/except` block** - feeds intentionally bad JSON (`"QUESTION"`, `confidence 95`, `"maybe"`) and catches the raised validation error, listing each violation.

**In one line:** declare the shape once, `model_validate_json` enforces types *and* value constraints together.

**What you'll see:** The good input prints category `question`, confidence `92%`, needs-reply `True`, and confirms `confidence is float`; the bad input is caught and the three violations (non-Literal category, confidence > 1, non-bool) are listed.

## 4 - Env vars & safe JSON parsing

Two production hygiene basics. First: read API keys from the environment, never hardcode them. Second: LLMs love to wrap JSON in markdown ```json fences, so a robust parser strips those before `json.loads`. This cell reads a key defensively and defines a fence-tolerant JSON parser, then tests it on fenced output.

In [ ]:
# =============================================
# secret management + JSON parsing
# Never hardcode API keys
# =============================================

import os, json

# ## Environment variables ##
# Set in terminal: export ANTHROPIC_API_KEY="sk-ant-..."
# Or in .env file with python-dotenv (pip install python-dotenv)

api_key = os.getenv("ANTHROPIC_API_KEY", "NOT_SET")
print(f"API key: {api_key[:8]}... (first 8 chars only)")

if api_key == "NOT_SET":
    print("Set your key: export ANTHROPIC_API_KEY='sk-ant-...'")

# ## Safely parse JSON from LLM output ##
# LLMs sometimes wrap JSON in ```json ... ``` fences
def safe_parse_json(text: str) -> dict:
    """Parse JSON from LLM output, handling markdown fences."""
    cleaned = text.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    if cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    try:
        return json.loads(cleaned.strip())
    except json.JSONDecodeError as e:
        print(f"Invalid JSON: {e}")
        return {}

# Test with messy LLM output
messy_output = '```json\n{"intent": "question", "score": 0.92}\n```'
parsed = safe_parse_json(messy_output)
print(f"\nParsed: {parsed}")
print(f"Intent: {parsed.get('intent')}")
print(f"Score:  {parsed.get('score')}")

# Output:
# API key: sk-ant-a... (first 8 chars only)
# 
# Parsed: {'intent': 'question', 'score': 0.92}
# Intent: question
# Score:  0.92

**Explanation**

A utilities cell - no model call, just the plumbing every app needs. It shows the safe `os.getenv` pattern with a default (so missing keys don't crash), and a small `safe_parse_json` helper that peels ```json / ``` fences and fails gracefully rather than throwing on malformed text.

**How the code works - step by step**
- **`os.getenv("ANTHROPIC_API_KEY", "NOT_SET")`** - reads the key with a fallback default, prints only the first 8 chars, and warns if it's unset.
- **`safe_parse_json`** - strips a leading ```json or ``` and a trailing ```, then `json.loads` the cleaned text inside a `try/except` that returns `{}` on a `JSONDecodeError`.
- **The test** - runs a fence-wrapped `messy_output` through the parser and reads back `intent` and `score` with `.get`.

**In one line:** read keys from the env, and strip fences before trusting LLM JSON.

**What you'll see:** Prints the masked key (`sk-ant-a...` or a `NOT_SET` warning if you have no key set), then the parsed dict `{'intent': 'question', 'score': 0.92}` with intent `question` and score `0.92`.

## 5 - Error handling & resilience: LLM APIs will fail

Rate limits (429), server errors (5xx), auth failures (401), and timeouts are routine at production scale, and each demands a different response. This cell builds the three-layer resilience pattern every serious LLM app uses: typed exceptions with per-error policy, exponential backoff with jitter, and a model fallback chain.

In [ ]:
# =============================================
# LLM error handling
# Pattern: structured errors + backoff + fallback chain
# =============================================

import time, random

# ## 1. Structured exception handling for LLM APIs ##
# Different errors need different responses:
#   429 Rate Limit  -> wait and retry (exponential backoff)
#   500 Server Error -> retry (might be temporary)
#   401 Auth Error  -> FAIL FAST (don't retry, key is wrong)
#   Timeout         -> retry with longer timeout

class RateLimitError(Exception): pass
class ServerError(Exception): pass
class AuthError(Exception): pass

def call_llm_unreliable(prompt: str) -> str:
    """Simulates an unreliable LLM API."""
    roll = random.random()
    if roll < 0.3: raise RateLimitError("429: Too Many Requests")
    if roll < 0.4: raise ServerError("500: Internal Server Error")
    if roll < 0.42: raise AuthError("401: Invalid API Key")
    time.sleep(0.5)
    return f"Response for: {prompt}"

# ## 2. Exponential backoff with jitter ##
def retry_with_backoff(func, prompt, max_retries=3):
    """Retry with exponential backoff. Fail fast on auth errors."""
    for attempt in range(max_retries):
        try:
            return func(prompt)
        except AuthError:
            print(f"  Auth error - FAIL FAST (don't retry)")
            raise  # Never retry auth errors
        except RateLimitError:
            wait = (2 ** attempt) + random.uniform(0, 1)  # jitter prevents thundering herd
            print(f"  Rate limited. Retry {attempt+1}/{max_retries} in {wait:.1f}s")
            time.sleep(wait)
        except ServerError:
            print(f"  Server error. Retry {attempt+1}/{max_retries}")
            time.sleep(1)
    raise Exception(f"Failed after {max_retries} retries")

# ## 3. Model fallback chain ##
def call_with_fallback(prompt: str) -> str:
    """Try primary, fall back to cheaper, then to cache."""
    models = ["claude-sonnet-4-6", "claude-haiku-4-6", "cached"]
    for model in models:
        try:
            print(f"  Trying {model}...")
            if model == "cached":
                return "[Cached fallback response]"
            return retry_with_backoff(call_llm_unreliable, prompt)
        except Exception as e:
            print(f"  {model} failed: {e}")
    return "Service temporarily unavailable"

# Test
print("Calling LLM with full error handling:\n")
result = call_with_fallback("What is async/await?")
print(f"\nFinal result: {result}")
print(f"\nIn production: use 'tenacity' library instead of hand-rolled retry")
print(f"  pip install tenacity")
print(f"  @retry(wait=wait_exponential(min=1, max=60), stop=stop_after_attempt(3))")

# Output:
# Calling LLM with full error handling:
#   Trying claude-sonnet-4-6...
#   Rate limited. Retry 1/3 in 1.4s
#   Server error. Retry 2/3
#   ...
#   Final result: Response for: What is async/await?

**Explanation**

A resilience harness around a deliberately flaky stub. `call_llm_unreliable` rolls a random number to raise different failure types. The two wrappers around it encode the actual production policy: fail fast on unrecoverable errors, back off and retry on transient ones, and degrade gracefully down a chain of cheaper options.

**How the code works - step by step**
- **Custom exception classes** - `RateLimitError`, `ServerError`, `AuthError` let the caller branch on failure *type*.
- **`call_llm_unreliable`** - `random.random()` decides the outcome: ~30% rate-limit, ~10% server error, ~2% auth error, else success after a short sleep.
- **`retry_with_backoff`** - re-raises `AuthError` immediately (never retry a bad key), sleeps `2**attempt + jitter` on rate limits, and retries server errors, giving up after `max_retries`.
- **`call_with_fallback`** - walks `[sonnet -> haiku -> cached]`, returning the first that succeeds; the final `cached` tier always yields a response.

**In one line:** fail fast on auth, back off (with jitter) on transient errors, fall back to a cheaper model or cache.

**What you'll see:** Because failures are random, output varies per run: you'll see `Trying claude-sonnet-4-6...` followed by some mix of rate-limit/server retry lines, then a final result (a real response, a fallback, or the cached tier), plus a tip to use the `tenacity` library in production.

## 6 - The production LLM call: all skills combined

This is the payoff - every pattern from the notebook fused into one pipeline, the exact shape of the LLM-call code you'll meet again in Modules 5-14. It reads the key from the env, validates output with Pydantic, caches with a decorator, runs calls in parallel with async, retries on failure, and streams each answer word by word.

> Runs in **demo mode** with simulated calls if `ANTHROPIC_API_KEY` is unset - no real key needed.

In [ ]:
# =============================================
# production LLM call - all 7 skills combined
# This is exactly how Modules 5-14 make LLM calls
# =============================================

import asyncio, os, json, time, functools
from pydantic import BaseModel, Field

# ## Skill 5: Env vars ##
API_KEY = os.getenv("ANTHROPIC_API_KEY", "demo-key")
if API_KEY == "demo-key":  # no real key set
    print("No ANTHROPIC_API_KEY set - running in demo mode (simulated calls).\n")

# ## Skill 3+4: Pydantic validates LLM output ##
class LLMResponse(BaseModel):
    answer: str = Field(..., min_length=1)
    confidence: float = Field(..., ge=0, le=1)
    tokens: int = Field(..., ge=1)

# ## Skill 4: Decorators stack (cache wraps async) ##
def cache(func):
    _c = {}
    @functools.wraps(func)
    async def wrapper(*a):
        k = str(a)
        if k in _c: return _c[k]
        r = await func(*a); _c[k] = r; return r
    return wrapper

# ## Skill 1+6: Async + error handling + retry ##
@cache
async def llm_call(prompt: str) -> LLMResponse:
    for attempt in range(3):
        try:
            await asyncio.sleep(0.5)  # simulated API call
            raw = {"answer": f"GenAI uses transformers for {prompt}",
                   "confidence": 0.92, "tokens": 45}
            return LLMResponse(**raw)  # Pydantic validates
        except Exception:
            if attempt < 2: await asyncio.sleep(2 ** attempt)
            else: raise

# ## Skill 2: Generator for streaming ##
def stream(text, delay=0.05):  # simulated 50ms per word
    for word in text.split():
        time.sleep(delay)
        yield word + " "

# ## Run the pipeline ##
async def main():
    prompts = ["RAG", "agents", "MCP"]

    # Parallel calls with async
    start = time.time()
    results = await asyncio.gather(*[llm_call(p) for p in prompts])
    print(f"3 calls in {time.time()-start:.1f}s (parallel)\n")

    # Repeat "RAG" - the first call already populated the cache,
    # so this returns instantly. (A plain dict cache only dedupes
    # AFTER a call finishes - concurrent duplicates in one gather()
    # still both run; real coalescing needs a lock or shared future.)
    start = time.time()
    cached = await llm_call("RAG")
    print(f"Repeat 'RAG' in {time.time()-start:.3f}s (from cache)\n")
    results.append(cached)

    # Stream each result
    for r in results:
        print(f"[{r.confidence:.0%} conf, {r.tokens} tokens] ", end="")
        for w in stream(r.answer): print(w, end="", flush=True)
        print()

await main()

# Output:
# No ANTHROPIC_API_KEY set - running in demo mode (simulated calls).
# 
# 3 calls in 0.5s (parallel)
# 
# Repeat 'RAG' in 0.000s (from cache)
# 
# [92% conf, 45 tokens] GenAI uses transformers for RAG
# [92% conf, 45 tokens] GenAI uses transformers for agents
# [92% conf, 45 tokens] GenAI uses transformers for MCP
# [92% conf, 45 tokens] GenAI uses transformers for RAG    # served from cache

**Explanation**

The capstone. Nothing new conceptually - it's the five earlier skills wired together into a realistic request path. Read it as a map: each commented `## Skill N` block marks where async, Pydantic, the cache decorator, retry, and the streaming generator plug in.

**How the code works - step by step**
- **`API_KEY = os.getenv(...)`** (env vars) - loads the key with a `demo-key` fallback and announces demo mode when unset.
- **`class LLMResponse(BaseModel)`** (Pydantic) - validates that every response has a non-empty answer, a 0-1 confidence, and a positive token count.
- **`cache` decorator** - wraps the async call in a dict-backed memo keyed by args, returning stored results instantly.
- **`llm_call`** (async + retry) - an `@cache`d coroutine that fakes the API with `asyncio.sleep`, retries up to 3 times with backoff, and returns a validated `LLMResponse`.
- **`stream`** (generator) - yields the answer word by word with a small delay to simulate token streaming.
- **`main`** - `asyncio.gather`s three prompts in parallel, then repeats `"RAG"` to hit the cache, then streams each result. The inline comment flags that a plain-dict cache only dedupes *after* a call finishes, so concurrent duplicates in one `gather` still both run.

**In one line:** env key -> parallel async calls -> Pydantic-validated -> cached -> streamed.

**What you'll see:** Prints the demo-mode notice, then `3 calls in ~0.5s (parallel)`, then the repeated `RAG` returning in ~0.000s from cache, then four streamed lines each tagged `[92% conf, 45 tokens]` - the last one served from cache.

You now have the six load-bearing Python patterns for GenAI in one place: async runs many I/O-bound LLM calls at once, generators stream tokens as they land, Pydantic turns messy model JSON into validated objects, decorators bolt on cache/retry/timing without touching business logic, env vars keep keys out of source, and layered error handling (fail-fast + backoff + fallback) keeps the app up when the API isn't. The capstone cell is the template every later lesson reuses. Next stop: Lesson 1.1 (Math Foundations) for the vectors and softmax behind model behavior, then Module 7 where these same patterns get productionized against real Anthropic/OpenAI clients.